[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/12_conv_and_vision_blocks.ipynb)

# 12. Convolution and vision blocks

Conv2d의 가장 기본 형태에서 grouped/depthwise/dilated와 residual vision block으로 확장한다.

**반복 형식:** 바닐라 PyTorch 실행 → profiler로 ATen/CUDA 연산 확인 → 필요할 때만 작은 텐서로 수학적 전개를 펼친다.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


In [ ]:
from torch.profiler import profile, ProfilerActivity

def profile_call(name, fn, *args, **kwargs):
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        out = fn(*args, **kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(f"\n[{name}] top operators")
    sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

    return out


## 1. Standard Conv2d

작은 이미지에 3x3 convolution을 적용한다.


In [ ]:
x = torch.arange(36, dtype=torch.float32, device=device).reshape(1, 1, 6, 6)
conv = nn.Conv2d(1, 2, kernel_size=3, padding=1, bias=False).to(device)
y = conv(x)
print("input:", x.shape, "output:", y.shape)


In [ ]:
_ = profile_call("Conv2d", conv, x)


## 2. Grouped convolution

channel을 독립 group으로 나눠 convolution한다.


In [ ]:
xg = torch.randn(1, 4, 8, 8, device=device)
grouped = nn.Conv2d(4, 4, 3, padding=1, groups=2, bias=False).to(device)
print(grouped(xg).shape)


In [ ]:
_ = profile_call("grouped conv", grouped, xg)


## 3. Depthwise + pointwise

channel별 spatial conv 뒤 1x1 conv로 channel을 섞는다.


In [ ]:
depthwise = nn.Conv2d(4, 4, 3, padding=1, groups=4, bias=False).to(device)
pointwise = nn.Conv2d(4, 8, 1, bias=False).to(device)

y = pointwise(depthwise(xg))
print(y.shape)


In [ ]:
_ = profile_call("depthwise", depthwise, xg)
_ = profile_call("depthwise + pointwise", lambda z: pointwise(depthwise(z)), xg)


## 4. Dilated convolution

kernel 사이 간격을 늘려 receptive field를 확장한다.


In [ ]:
dilated = nn.Conv2d(4, 4, 3, padding=2, dilation=2, bias=False).to(device)
print(dilated(xg).shape)


In [ ]:
_ = profile_call("dilated conv", dilated, xg)


## 5. ResNet bottleneck

1x1 → 3x3 → 1x1과 identity shortcut을 결합한다.


In [ ]:
class TinyBottleneck(nn.Module):
    def __init__(self, c=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(c, c // 2, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(c // 2, c // 2, 3, padding=1, bias=False),
            nn.ReLU(),
            nn.Conv2d(c // 2, c, 1, bias=False),
        )

    def forward(self, x):
        return x + self.net(x)

xb = torch.randn(1, 8, 8, 8, device=device)
block = TinyBottleneck().to(device)
print(block(xb).shape)


In [ ]:
_ = profile_call("ResNet bottleneck", block, xb)


## 6. ConvNeXt-style block

depthwise conv + channel-last MLP를 결합한다.


In [ ]:
class TinyConvNeXt(nn.Module):
    def __init__(self, c=8):
        super().__init__()
        self.dw = nn.Conv2d(c, c, 7, padding=3, groups=c)
        self.norm = nn.LayerNorm(c)
        self.pw1 = nn.Linear(c, 4 * c)
        self.pw2 = nn.Linear(4 * c, c)

    def forward(self, x):
        residual = x
        x = self.dw(x).permute(0, 2, 3, 1)
        x = self.pw2(F.gelu(self.pw1(self.norm(x))))
        return residual + x.permute(0, 3, 1, 2)

cnx = TinyConvNeXt().to(device)
print(cnx(xb).shape)


In [ ]:
_ = profile_call("ConvNeXt block", cnx, xb)


## References and provenance

**[12.1] ResNet**
- 출처: He et al., Deep Residual Learning
- 이 노트북에서 가져온 부분: residual bottleneck

**[12.2] MobileNet**
- 출처: Howard et al., MobileNets
- 이 노트북에서 가져온 부분: depthwise separable convolution

**[12.3] ConvNeXt**
- 출처: Liu et al., A ConvNet for the 2020s
- 이 노트북에서 가져온 부분: depthwise conv + modern MLP block
